In [6]:
import pandas as pd

# 1. Load your two tables
df1 = pd.read_csv('/home/lq53/mir_repos/dappy_24_nov/mir_modif_dappy/meta_datas/250710Oct3V1.csv')
df2 = pd.read_csv('/home/lq53/mir_repos/dappy_24_nov/mir_modif_dappy/meta_datas/sum4folder_20250710.csv')

# 2. Outer merge on the key, with suffixes so overlapping columns show up as *_1 and *_2
key = 'Prediction_path'
merged = pd.merge(df1, df2, on=key, how='outer', suffixes=('_1', '_2'))

# 3. Identify columns present in both (besides the key)
common = [c for c in df1.columns if c in df2.columns and c != key]

# 4. For each common column, pick df1’s value when present, else df2’s
for c in common:
    merged[c] = merged[f'{c}_1'].combine_first(merged[f'{c}_2'])
    merged.drop([f'{c}_1', f'{c}_2'], axis=1, inplace=True)

# 5. Fill any remaining NaNs with 0
merged.fillna(0, inplace=True)

# 6. Determine which columns came only from df2
extra = [c for c in df2.columns if c not in df1.columns]

# 7. Reorder so that:
#    – columns follow df1.columns exactly (key + the rest)
#    – then all df2-only columns in df2’s original sequence
final_cols = df1.columns.tolist() + extra
merged = merged[final_cols]

# 8. Save
out_path = '/home/lq53/mir_repos/dappy_24_nov/mir_modif_dappy/meta_datas/merged_full_250710.csv'
merged.to_csv(out_path, index=False)

print("Merged shape:", merged.shape)


Merged shape: (148, 17)
